In [ ]:
# PARAMETERS: Values the pipeline can override when it runs this notebook (this cell is tagged as the parameter cell).
source_name = "all"       # "all" loads every source; or a single source name such as "accounts"
batch_id = "manual"       # run identifier, stamped on every audit row
run_maintenance = True    # run OPTIMIZE on the Silver tables at the end

In [ ]:
# SETUP: Make the shared helper module importable, load its functions, and define the contracts file path.
import sys, json
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sys.path.append("/lakehouse/default/Files/config")                      # lets Python find silver_helpers.py
from silver_helpers import load_contract, validate_and_split, reconcile

CONTRACTS = "/lakehouse/default/Files/config/silver_contracts.json"     # the rulebook for all nine sources

# Bronze is not this notebook's default lakehouse, so read its tables directly from their OneLake path
BRONZE = "abfss://82b494dc-d048-486d-9dc1-61aed01f62a3@onelake.dfs.fabric.microsoft.com/031b0202-314e-4fa8-a994-e2fccfc85bea/Tables/dbo"

def bronze(table_name):
    return spark.read.format("delta").load(f"{BRONZE}/{table_name}")   # load one Bronze Delta table as a DataFrame


In [ ]:
# WRITE FUNCTION: write_silver() merges a source's valid rows into its Silver table (update if the key
# exists, insert if not, so reruns do not duplicate) and refreshes that source's rows in slv_quarantine.
def write_silver(valid, quarantine, contract):
    table = contract["silver_table"]       # e.g. slv_accounts
    pk = contract["primary_key"]           # key columns that identify one row

    if not spark.catalog.tableExists(table):                      # first run only: create an empty table
        valid.limit(0).write.format("delta").saveAsTable(table)

    valid.createOrReplaceTempView("incoming")                     # name the good rows so SQL can read them
    on = " AND ".join(f"t.`{k}` <=> s.`{k}`" for k in pk)         # <=> is equals that treats NULL = NULL
    spark.sql(f"""
        MERGE INTO {table} t USING incoming s ON {on}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    if spark.catalog.tableExists("slv_quarantine"):               # quarantine is rebuilt per source each run
        spark.sql(f"DELETE FROM slv_quarantine WHERE source = '{contract['source']}'")
    quarantine.write.format("delta").mode("append").saveAsTable("slv_quarantine")

In [ ]:
# LOAD SOURCES: For each source, validate the Bronze table, merge good rows into Silver, send bad rows
# to quarantine, reconcile row counts against Bronze, and record one row per source in slv_load_audit.
all_sources = list(json.load(open(CONTRACTS))["sources"])
to_run = all_sources if source_name == "all" else [source_name]

audit = []
for name in to_run:
    try:
        c = load_contract(CONTRACTS, name)                         # rules for this source
        b = bronze(c["bronze_table"])                              # its raw Bronze table, read by OneLake path
        v, q = validate_and_split(b, c)                            # valid rows / quarantined rows
        write_silver(v, q, c)                                      # merge valid, save quarantined
        r = reconcile(b, v, q)                                     # rows in must equal valid + quarantined
        audit.append((batch_id, name, r["rows_in"], r["rows_valid"], r["rows_quarantined"],
                      bool(r["balanced"]), "Succeeded", None))
    except Exception as e:
        audit.append((batch_id, name, None, None, None, None, "Failed", str(e)[:1000]))

audit_schema = ("batch_id string, source string, rows_in long, rows_valid long, rows_quarantined long, "
                "balanced boolean, status string, error_message string")
audit_df = spark.createDataFrame(audit, audit_schema).withColumn("loaded_at", F.current_timestamp())
audit_df.write.format("delta").mode("append").saveAsTable("slv_load_audit")   # append: a log keeps history
display(audit_df)

In [ ]:
# INVOICE LINES: Split each invoice's nested `lines` array into one row per line item and merge them
# into slv_invoice_lines (key = invoice_id + line_id). Runs only when invoices were loaded.
if "invoices" in to_run:
    w = Window.partitionBy("invoice_id").orderBy(F.col("modified_at").desc_nulls_last())
    latest = bronze("brz_invoices").withColumn("rn", F.row_number().over(w)).filter("rn = 1")   # newest copy per invoice
    valid_ids = spark.table("slv_invoices").select("invoice_id").distinct()                          # validated invoices only

    lines = (latest
        .select("invoice_id", F.explode("lines").alias("l"))                 # one row per array element
        .select("invoice_id", "l.line_id", "l.product",
                F.col("l.quantity").cast("int").alias("quantity"),
                F.col("l.unit_price").alias("unit_price"),
                F.col("l.amount").alias("amount"),
                F.col("l.period_start").cast("date").alias("period_start"),
                F.col("l.period_end").cast("date").alias("period_end"))
        .join(valid_ids, "invoice_id", "inner"))

    if not spark.catalog.tableExists("slv_invoice_lines"):
        lines.limit(0).write.format("delta").saveAsTable("slv_invoice_lines")
    lines.createOrReplaceTempView("incoming_lines")
    spark.sql("""
        MERGE INTO slv_invoice_lines t USING incoming_lines s
        ON t.invoice_id <=> s.invoice_id AND t.line_id <=> s.line_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print("slv_invoice_lines rows:", spark.table("slv_invoice_lines").count())

In [ ]:
# SCD2 ACCOUNTS: Rebuild slv_account_history from the account snapshots. A new version row starts only
# when tier, account_status or owner changes. Runs only when accounts were loaded.
if "accounts" in to_run:
    tracked = ["tier", "account_status", "owner"]                           # attributes we keep history for
    w = Window.partitionBy("account_id").orderBy("snapshot_date")           # each account's snapshots in date order

    a = spark.table("slv_accounts")
    a = a.withColumn("_h", F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("<null>")) for c in tracked]), 256))
    a = (a.withColumn("_prev_h", F.lag("_h").over(w))                       # keep a snapshot if it is the first,
           .filter(F.col("_prev_h").isNull() | (F.col("_h") != F.col("_prev_h"))))   # or the fingerprint changed
    a = (a.withColumn("valid_from", F.col("snapshot_date"))
           .withColumn("valid_to", F.lead("snapshot_date").over(w))         # next version's start; NULL for the latest
           .withColumn("is_current", F.col("valid_to").isNull()))

    dim = a.select("account_id", "account_name", "industry", "country", "tier", "account_status",
                   "owner", "created_date", "acquisition_campaign_id", "valid_from", "valid_to", "is_current")
    dim.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("slv_account_history")
    print("slv_account_history rows:", spark.table("slv_account_history").count())

In [ ]:
# FK CHECKS: Count orphan rows (child keys with no matching parent) for each relationship and save one
# row per relationship per run in slv_dq_results. Runs only on a full load.
def run_fk_checks(relationships, run_id):
    rows = []
    for child, ccol, parent, pcol in relationships:
        c = spark.table(child).filter(F.col(ccol).isNotNull())                  # child rows that have a key
        p = spark.table(parent).select(F.col(pcol).alias("_pk")).distinct()     # parent keys
        orphans = c.join(p, F.col(ccol) == F.col("_pk"), "left_anti").count()   # child rows with no parent
        rows.append((run_id, "fk_" + child + "_" + ccol, child, parent, c.count(), orphans))
    out = (spark.createDataFrame(rows, "run_id string, check_name string, child_table string, "
                                       "parent_table string, rows_checked long, orphan_rows long")
           .withColumn("checked_at", F.current_timestamp()))
    out.write.format("delta").mode("append").saveAsTable("slv_dq_results")      # append: a log keeps history
    return out

relationships = [
    ("slv_contacts",      "account_id",      "slv_accounts",      "account_id"),
    ("slv_opportunities", "account_id",      "slv_accounts",      "account_id"),
    ("slv_subscriptions", "account_id",      "slv_accounts",      "account_id"),
    ("slv_invoices",      "account_id",      "slv_accounts",      "account_id"),
    ("slv_invoices",      "subscription_id", "slv_subscriptions", "subscription_id"),
    ("slv_usage_events",  "account_id",      "slv_accounts",      "account_id"),
    ("slv_web_sessions",  "account_id",      "slv_accounts",      "account_id"),
    ("slv_ad_spend",      "campaign_id",     "slv_campaigns",     "campaign_id"),
]
if source_name == "all":
    display(run_fk_checks(relationships, batch_id))

In [ ]:
# MAINTENANCE: Compact small files in every Silver table with OPTIMIZE (VACUUM stays at the 7-day default
# and is not run here). Controlled by the run_maintenance parameter.
if run_maintenance:
    silver_tables = [t.name for t in spark.catalog.listTables() if t.name.startswith("slv_")]
    for t in silver_tables:
        spark.sql(f"OPTIMIZE {t}")
    print("optimized:", silver_tables)

In [ ]:
# FINAL CHECK: Fail the notebook (so the pipeline run shows as failed) if any source failed or its row
# counts did not reconcile; details are in slv_load_audit for this batch_id.
bad = audit_df.filter("status != 'Succeeded' OR balanced = false").count()
if bad:
    raise Exception(f"{bad} source(s) failed or did not reconcile; see slv_load_audit for batch {batch_id}")
print("Silver load complete for batch", batch_id)